<div style="background: linear-gradient(135deg, #7B4F00 0%, #f5a623 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #fff3d6; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #fff3d6; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: white; font-size: 1.05em; font-weight: 600;">&#9733; Bonus Chapter &nbsp;&middot;&nbsp; From Notebook to Production</span>
  </div>
</div>
<div style="background: #fff8ec; border-left: 5px solid #f5a623; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>Once a model is trained and saved, how do you make it available to other people?
  This chapter takes the trained models from Chapters 4 and 5 and deploys them as
  permanent, free web applications on Hugging Face Spaces &mdash; no server, no Docker,
  no command line required.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| B.1 The Gap Between Notebook and Production | Why a notebook is not enough · What a deployed model looks like |
| B.2 The Deployment Stack | Gradio · Hugging Face Spaces · How the three pieces fit together |
| B.3 Deploying the Churn Classifier (FFN) | Writing `app.py` · Uploading to Spaces · Testing the live app |
| B.4 Deploying the Defect Detector (CNN) | Image upload interface · PIL preprocessing · Live predictions |
| B.5 Updating a Deployed Model | Uploading a new `.pth` file · When predictions change |

> **Prerequisites:** Chapter 4 (churn classifier) and Chapter 5 (defect detector CNN).
> The `.pth` files saved in those chapters are what we deploy here.
>
> **Everything is free.** Hugging Face Spaces provides permanent free hosting.
> You only need a browser — no Docker, no terminal, no installation on your computer.

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Bonus Chapter — Setup
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet gradio torch torchvision scikit-learn dill numpy pillow

import torch, numpy as np, dill, gradio as gr
from PIL import Image

print(f"Gradio  : {gr.__version__}")
print(f"PyTorch : {torch.__version__}")
print("Setup complete.")

---
## Where We Are Coming From

In Chapters 4 and 5 you trained two models and saved them using `ModelPipeline`:

```
churn_model_v1.pth      (Chapter 4 — FFN classifier: will a customer churn?)
defect_model_v1.pth     (Chapter 5 — CNN classifier: is this product defective?)
```

Each file is self-contained: weights, scaler, feature names, architecture config —
everything bundled together so the model can run on any machine.

**The problem:** only you can use them right now.

A product manager, a quality engineer, or a customer success team cannot open a
Jupyter notebook. They need a form they can fill in, or an image they can upload,
and a result they can act on.

**This chapter closes that gap — permanently and for free.**

By the end you will have two live web apps:

```
https://huggingface.co/spaces/your-username/churn-predictor
https://huggingface.co/spaces/your-username/defect-detector
```

Anyone with those links can use your models from any device, any browser, anywhere.
The apps stay live as long as your Hugging Face account exists.

> **Before starting:** make sure you have downloaded `churn_model_v1.pth` from your
> Chapter 4 session and `defect_model_v1.pth` from your Chapter 5 session.
> Each section in this chapter has an upload cell that walks you through this step.

---
# B.1 The Gap Between Notebook and Production

## What a notebook cannot do

A Jupyter notebook is a document. It requires a human to open it, run it, and read it.
No other person or program can send data to a notebook and get a result back automatically.

| | Notebook | Deployed app |
|-|----------|--------------|
| Who can use it | Only you, with the notebook open | Anyone with the URL |
| Device needed | Laptop with Python | Any browser — phone, tablet, PC |
| Requires technical knowledge | Yes | No |
| Always available | Only when your laptop is on | 24/7 |
| Another system can call it | No | Yes |

## What we are building

A non-technical user opens a URL and sees this:

```
Churn Predictor
───────────────────────────────────────────────
Tenure (months)          [ 12        ]
Monthly Charges ($)      [ 75.00     ]
Total Charges ($)        [ 900.00    ]
Number of Products       [ 2         ]
Has Internet Service     [x] Yes
Has Phone Service        [x] Yes

                   [ Predict ]

Result:  Churn Probability: 68%
         Risk Level: High — recommend a retention call
───────────────────────────────────────────────
```

They fill in the fields, click Predict, and get an answer.
No Python. No notebook. No setup. The model is running on Hugging Face's servers,
not on your laptop.

---
# B.2 The Deployment Stack

Three pieces work together. You only need to write the code — the other two
are free services that handle everything else.

```
┌──────────────────────────────────────────────────────────────┐
│                                                               │
│   Your .pth file          Gradio               HF Spaces     │
│   (trained model)  ──>  (web interface) ──>  (free hosting)  │
│                                                               │
│   Holds the               Turns your          Runs your app  │
│   pipeline: weights,      predict()           permanently    │
│   scaler, features        function into       on the internet│
│                           a web form                         │
└──────────────────────────────────────────────────────────────┘
```

---

## Gradio — you already know this

You used Gradio in Chapter 7 to build a chat interface.
Here you use it for a prediction form — the same idea, different input components.

```python
# Chapter 7 (chat):
gr.ChatInterface(fn=respond)

# Bonus Chapter (prediction form):
gr.Interface(fn=predict, inputs=[...], outputs=[...])
```

`gr.Interface()` takes your prediction function and a list of input components
(one per feature) and builds the form automatically.

---

## Hugging Face Spaces — permanent free hosting

Hugging Face Spaces hosts Gradio apps for free, indefinitely.

To deploy, you upload exactly **three files** to a Space:

```
your-space/
    app.py               ← your Gradio application  (we write this in B.3 and B.4)
    your_model.pth       ← the trained pipeline from Chapter 4 or 5
    requirements.txt     ← the Python packages the Space needs to install
```

Hugging Face reads those files, installs the packages, and runs `app.py`.
Your app is live at `https://huggingface.co/spaces/username/space-name`.

No command line. No Docker. No server configuration.
You upload files through a browser — like attaching files to an email.

---
# B.3 Deploying the Churn Classifier (FFN)

## Step 1 — Write `app.py` locally and test it here

The cell below writes `app.py` to disk and launches a local preview.
Test it in the notebook first — then upload to Hugging Face.

## Before you start — upload your model file

The churn model was saved as `churn_model_v1.pth` at the end of Chapter 4.
You need to upload it to this Colab session before any code in this section will run.

> **Where is the file?**
> If you ran Chapter 4 in Colab, the file was saved inside that session.
> Colab sessions do not share files — you need to download it from the Chapter 4
> session first, then upload it here.
>
> **To download from your Chapter 4 session:**
> In that notebook, run:
> ```python
> from google.colab import files
> files.download("churn_model_v1.pth")
> ```
> The file will appear in your browser's Downloads folder.
>
> **Then upload it here using the cell below.**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Upload churn_model_v1.pth to this Colab session
#
# Click the "Choose Files" button that appears after running this cell.
# Select churn_model_v1.pth from your Downloads folder.
# Wait for the upload bar to complete before running any cells below.
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import files
import os

print("Click 'Choose Files' and select churn_model_v1.pth")
print()

uploaded = files.upload()

# Confirm the right file was uploaded
if "churn_model_v1.pth" in uploaded:
    size_mb = os.path.getsize("churn_model_v1.pth") / 1024 / 1024
    print(f"churn_model_v1.pth uploaded successfully  ({size_mb:.1f} MB)")
    print("You can now run the cells below.")
else:
    print("WARNING: the uploaded file is not named churn_model_v1.pth")
    print(f"  Files received: {list(uploaded.keys())}")
    print("  Rename the file to churn_model_v1.pth and re-run this cell.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write app.py for the churn classifier
# This is the complete file you will upload to Hugging Face Spaces.
# ─────────────────────────────────────────────────────────────────────────────

app_content = 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport numpy as np\nimport pandas as pd\nimport dill\nimport inspect\nimport gradio as gr\n\n# ── Feature encoding maps ──────────────────────────────────────────────────────\n# Chapter 4 uses LabelEncoder on all categorical columns.\n# LabelEncoder assigns codes in alphabetical order.\n# These maps convert human-readable dropdown values to the same integer codes.\n\nENCODE = {\n    "gender":          {"Female": 0, "Male": 1},\n    "SeniorCitizen":   {"No": 0, "Yes": 1},\n    "Partner":         {"No": 0, "Yes": 1},\n    "Dependents":      {"No": 0, "Yes": 1},\n    "PhoneService":    {"No": 0, "Yes": 1},\n    "MultipleLines":   {"No": 0, "No phone service": 1, "Yes": 2},\n    "InternetService": {"DSL": 0, "Fiber optic": 1, "No": 2},\n    "OnlineSecurity":  {"No": 0, "No internet service": 1, "Yes": 2},\n    "OnlineBackup":    {"No": 0, "No internet service": 1, "Yes": 2},\n    "DeviceProtection":{"No": 0, "No internet service": 1, "Yes": 2},\n    "TechSupport":     {"No": 0, "No internet service": 1, "Yes": 2},\n    "StreamingTV":     {"No": 0, "No internet service": 1, "Yes": 2},\n    "StreamingMovies": {"No": 0, "No internet service": 1, "Yes": 2},\n    "Contract":        {"Month-to-month": 0, "One year": 1, "Two year": 2},\n    "PaperlessBilling":{"No": 0, "Yes": 1},\n    "PaymentMethod":   {\n        "Bank transfer (automatic)": 0,\n        "Credit card (automatic)":   1,\n        "Electronic check":          2,\n        "Mailed check":              3,\n    },\n}\n\n\n# ── ModelPipeline (load-only) ──────────────────────────────────────────────────\n# Matches the full Chapter 4 ModelPipeline interface.\n# predict() accepts a pandas DataFrame and returns a DataFrame — same contract.\n\nclass ModelPipeline:\n    def __init__(self):\n        self.model         = None\n        self.preprocessor  = None\n        self.feature_names = []\n        self.feature_ranges = {}\n        self.model_config  = {}\n\n    @classmethod\n    def load(cls, path):\n        ckpt   = torch.load(path, map_location=torch.device("cpu"), weights_only=False)\n        MC_raw = dill.loads(ckpt["model_class_bytes"])\n        try:\n            src = dill.source.getsource(MC_raw)\n            ns  = {"torch": torch, "nn": nn, "F": F, "np": np}\n            exec(compile(src, "<string>", "exec"), ns)\n            MC  = ns[MC_raw.__name__]\n        except Exception:\n            MC = MC_raw\n        valid = set(inspect.signature(MC.__init__).parameters) - {"self"}\n        cfg   = {k: v for k, v in ckpt["model_config"].items() if k in valid}\n        obj               = cls()\n        obj.model         = MC(**cfg)\n        obj.model.load_state_dict(ckpt["state_dict"])\n        obj.model.eval()\n        obj.preprocessor   = ckpt.get("preprocessor")\n        obj.feature_names  = ckpt.get("feature_names", [])\n        obj.feature_ranges = ckpt.get("feature_ranges", {})\n        obj.model_config   = ckpt.get("model_config", {})\n        return obj\n\n    def predict(self, X_df, threshold=0.5):\n        # Step 1: validate columns\n        missing = set(self.feature_names) - set(X_df.columns)\n        if missing:\n            raise ValueError(f"Missing columns: {sorted(missing)}")\n        # Step 2: reorder columns to match training order, scale, predict\n        X = self.preprocessor.transform(\n            X_df[self.feature_names].to_numpy().astype(np.float32)\n        )\n        self.model.eval()\n        with torch.no_grad():\n            probs = torch.softmax(\n                self.model(torch.tensor(X, dtype=torch.float32)), dim=1\n            )[:, 1].cpu().numpy()\n        class_labels = self.model_config.get("class_labels", {0: "No Churn", 1: "Churn"})\n        return pd.DataFrame({\n            "churn_probability": np.round(probs, 4),\n            "predicted_label":   [class_labels[int(p >= threshold)] for p in probs],\n        })\n\n\npipeline = ModelPipeline.load("churn_model_v1.pth")\n\n\n# ── Prediction function ────────────────────────────────────────────────────────\ndef predict_churn(gender, senior, partner, dependents, tenure,\n                  phone, multiline, internet, security, backup,\n                  device, techsupport, tv, movies,\n                  contract, paperless, payment,\n                  monthly, total):\n\n    # Build a single-row DataFrame — same structure as Chapter 4 training data\n    row = {\n        "gender":          ENCODE["gender"][gender],\n        "SeniorCitizen":   ENCODE["SeniorCitizen"][senior],\n        "Partner":         ENCODE["Partner"][partner],\n        "Dependents":      ENCODE["Dependents"][dependents],\n        "tenure":          float(tenure),\n        "PhoneService":    ENCODE["PhoneService"][phone],\n        "MultipleLines":   ENCODE["MultipleLines"][multiline],\n        "InternetService": ENCODE["InternetService"][internet],\n        "OnlineSecurity":  ENCODE["OnlineSecurity"][security],\n        "OnlineBackup":    ENCODE["OnlineBackup"][backup],\n        "DeviceProtection":ENCODE["DeviceProtection"][device],\n        "TechSupport":     ENCODE["TechSupport"][techsupport],\n        "StreamingTV":     ENCODE["StreamingTV"][tv],\n        "StreamingMovies": ENCODE["StreamingMovies"][movies],\n        "Contract":        ENCODE["Contract"][contract],\n        "PaperlessBilling":ENCODE["PaperlessBilling"][paperless],\n        "PaymentMethod":   ENCODE["PaymentMethod"][payment],\n        "MonthlyCharges":  float(monthly),\n        "TotalCharges":    float(total),\n    }\n\n    # Pass a DataFrame to pipeline.predict() — exactly as Chapter 4 intended\n    df_input = pd.DataFrame([row])\n    result   = pipeline.predict(df_input, threshold=0.5)\n\n    prob = float(result["churn_probability"].iloc[0])\n    if prob >= 0.70:\n        risk = "High risk  --  recommend a retention call immediately."\n    elif prob >= 0.40:\n        risk = "Medium risk  --  monitor and consider a proactive check-in."\n    else:\n        risk = "Low risk  --  customer appears stable."\n\n    return f"{prob:.0%}", risk\n\n\n# ── Gradio interface ────────────────────────────────────────────────────────────\ndemo = gr.Interface(\n    fn = predict_churn,\n    inputs = [\n        gr.Dropdown(["Female", "Male"],                                         label="Gender"),\n        gr.Dropdown(["No", "Yes"],                                              label="Senior Citizen"),\n        gr.Dropdown(["No", "Yes"],                                              label="Partner"),\n        gr.Dropdown(["No", "Yes"],                                              label="Dependents"),\n        gr.Slider(0, 72, step=1, value=12,                                      label="Tenure (months)"),\n        gr.Dropdown(["No", "Yes"],                                              label="Phone Service"),\n        gr.Dropdown(["No", "No phone service", "Yes"],                         label="Multiple Lines"),\n        gr.Dropdown(["DSL", "Fiber optic", "No"],                              label="Internet Service"),\n        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Online Security"),\n        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Online Backup"),\n        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Device Protection"),\n        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Tech Support"),\n        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Streaming TV"),\n        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Streaming Movies"),\n        gr.Dropdown(["Month-to-month", "One year", "Two year"],                label="Contract"),\n        gr.Dropdown(["No", "Yes"],                                              label="Paperless Billing"),\n        gr.Dropdown(["Bank transfer (automatic)", "Credit card (automatic)",\n                     "Electronic check", "Mailed check"],                      label="Payment Method"),\n        gr.Slider(18, 120, step=0.5, value=65.0,                               label="Monthly Charges ($)"),\n        gr.Number(value=780.0,                                                  label="Total Charges ($)"),\n    ],\n    outputs = [\n        gr.Text(label="Churn Probability"),\n        gr.Text(label="Risk Assessment"),\n    ],\n    title       = "Customer Churn Predictor",\n    description = "Enter customer account details to predict churn likelihood.",\n    examples    = [\n        ["Male",   "No", "Yes", "No",  12, "Yes", "No",               "Fiber optic",\n         "No", "No", "No", "No", "No", "No",\n         "Month-to-month", "Yes", "Electronic check",          75.0,  900.0],\n        ["Female", "No", "Yes", "Yes", 60, "Yes", "Yes",              "DSL",\n         "Yes","Yes","Yes","Yes","Yes","Yes",\n         "Two year",       "No",  "Bank transfer (automatic)", 45.0, 2700.0],\n        ["Male",   "Yes","No",  "No",  2,  "Yes", "No phone service", "Fiber optic",\n         "No", "No", "No", "No", "Yes","Yes",\n         "Month-to-month", "Yes", "Electronic check",          95.0,  190.0],\n    ],\n    allow_flagging = "never",\n)\n\nif __name__ == "__main__":\n    demo.launch()\n'

with open("app.py", "w") as f:
    f.write(app_content)

print("app.py written.")
print(f"  Lines: {len(app_content.splitlines())}")

from google.colab import files
files.download("app.py")
print("app.py downloaded — upload to your Hugging Face Space.")

### Preview the app locally

Run the cell below to see the churn app inside this notebook before uploading.
`share=False` keeps it local — no public URL yet.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Local preview of the churn app
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import dill
import inspect
import gradio as gr

# ── Feature encoding maps ──────────────────────────────────────────────────────
# Chapter 4 uses LabelEncoder on all categorical columns.
# LabelEncoder assigns codes in alphabetical order.
# These maps convert human-readable dropdown values to the same integer codes.

ENCODE = {
    "gender":          {"Female": 0, "Male": 1},
    "SeniorCitizen":   {"No": 0, "Yes": 1},
    "Partner":         {"No": 0, "Yes": 1},
    "Dependents":      {"No": 0, "Yes": 1},
    "PhoneService":    {"No": 0, "Yes": 1},
    "MultipleLines":   {"No": 0, "No phone service": 1, "Yes": 2},
    "InternetService": {"DSL": 0, "Fiber optic": 1, "No": 2},
    "OnlineSecurity":  {"No": 0, "No internet service": 1, "Yes": 2},
    "OnlineBackup":    {"No": 0, "No internet service": 1, "Yes": 2},
    "DeviceProtection":{"No": 0, "No internet service": 1, "Yes": 2},
    "TechSupport":     {"No": 0, "No internet service": 1, "Yes": 2},
    "StreamingTV":     {"No": 0, "No internet service": 1, "Yes": 2},
    "StreamingMovies": {"No": 0, "No internet service": 1, "Yes": 2},
    "Contract":        {"Month-to-month": 0, "One year": 1, "Two year": 2},
    "PaperlessBilling":{"No": 0, "Yes": 1},
    "PaymentMethod":   {
        "Bank transfer (automatic)": 0,
        "Credit card (automatic)":   1,
        "Electronic check":          2,
        "Mailed check":              3,
    },
}


# ── ModelPipeline (load-only) ──────────────────────────────────────────────────
# Matches the full Chapter 4 ModelPipeline interface.
# predict() accepts a pandas DataFrame and returns a DataFrame — same contract.

class ModelPipeline:
    def __init__(self):
        self.model         = None
        self.preprocessor  = None
        self.feature_names = []
        self.feature_ranges = {}
        self.model_config  = {}

    @classmethod
    def load(cls, path):
        ckpt   = torch.load(path, map_location=torch.device("cpu"), weights_only=False)
        MC_raw = dill.loads(ckpt["model_class_bytes"])
        try:
            src = dill.source.getsource(MC_raw)
            ns  = {"torch": torch, "nn": nn, "F": F, "np": np}
            exec(compile(src, "<string>", "exec"), ns)
            MC  = ns[MC_raw.__name__]
        except Exception:
            MC = MC_raw
        valid = set(inspect.signature(MC.__init__).parameters) - {"self"}
        cfg   = {k: v for k, v in ckpt["model_config"].items() if k in valid}
        obj               = cls()
        obj.model         = MC(**cfg)
        obj.model.load_state_dict(ckpt["state_dict"])
        obj.model.eval()
        obj.preprocessor   = ckpt.get("preprocessor")
        obj.feature_names  = ckpt.get("feature_names", [])
        obj.feature_ranges = ckpt.get("feature_ranges", {})
        obj.model_config   = ckpt.get("model_config", {})
        return obj

    def predict(self, X_df, threshold=0.5):
        # Step 1: validate columns
        missing = set(self.feature_names) - set(X_df.columns)
        if missing:
            raise ValueError(f"Missing columns: {sorted(missing)}")
        # Step 2: reorder columns to match training order, scale, predict
        X = self.preprocessor.transform(
            X_df[self.feature_names].to_numpy().astype(np.float32)
        )
        self.model.eval()
        with torch.no_grad():
            probs = torch.softmax(
                self.model(torch.tensor(X, dtype=torch.float32)), dim=1
            )[:, 1].cpu().numpy()
        class_labels = self.model_config.get("class_labels", {0: "No Churn", 1: "Churn"})
        return pd.DataFrame({
            "churn_probability": np.round(probs, 4),
            "predicted_label":   [class_labels[int(p >= threshold)] for p in probs],
        })


pipeline = ModelPipeline.load("churn_model_v1.pth")
print(f"Pipeline loaded. Features: {pipeline.feature_names}")

# Quick sanity test — uses pipeline.predict(DataFrame) exactly as Chapter 4
test_row = {
    "gender": ENCODE["gender"]["Male"],
    "SeniorCitizen": ENCODE["SeniorCitizen"]["No"],
    "Partner": ENCODE["Partner"]["Yes"],
    "Dependents": ENCODE["Dependents"]["No"],
    "tenure": 12.0,
    "PhoneService": ENCODE["PhoneService"]["Yes"],
    "MultipleLines": ENCODE["MultipleLines"]["No"],
    "InternetService": ENCODE["InternetService"]["Fiber optic"],
    "OnlineSecurity": ENCODE["OnlineSecurity"]["No"],
    "OnlineBackup": ENCODE["OnlineBackup"]["No"],
    "DeviceProtection": ENCODE["DeviceProtection"]["No"],
    "TechSupport": ENCODE["TechSupport"]["No"],
    "StreamingTV": ENCODE["StreamingTV"]["No"],
    "StreamingMovies": ENCODE["StreamingMovies"]["No"],
    "Contract": ENCODE["Contract"]["Month-to-month"],
    "PaperlessBilling": ENCODE["PaperlessBilling"]["Yes"],
    "PaymentMethod": ENCODE["PaymentMethod"]["Electronic check"],
    "MonthlyCharges": 75.0,
    "TotalCharges": 900.0,
}
test_df = pd.DataFrame([test_row])
test_result = pipeline.predict(test_df, threshold=0.5)
print(f"Test result:")
print(test_result)



# ── Prediction function ────────────────────────────────────────────────────────
def predict_churn(gender, senior, partner, dependents, tenure,
                  phone, multiline, internet, security, backup,
                  device, techsupport, tv, movies,
                  contract, paperless, payment,
                  monthly, total):

    # Build a single-row DataFrame — same structure as Chapter 4 training data
    row = {
        "gender":          ENCODE["gender"][gender],
        "SeniorCitizen":   ENCODE["SeniorCitizen"][senior],
        "Partner":         ENCODE["Partner"][partner],
        "Dependents":      ENCODE["Dependents"][dependents],
        "tenure":          float(tenure),
        "PhoneService":    ENCODE["PhoneService"][phone],
        "MultipleLines":   ENCODE["MultipleLines"][multiline],
        "InternetService": ENCODE["InternetService"][internet],
        "OnlineSecurity":  ENCODE["OnlineSecurity"][security],
        "OnlineBackup":    ENCODE["OnlineBackup"][backup],
        "DeviceProtection":ENCODE["DeviceProtection"][device],
        "TechSupport":     ENCODE["TechSupport"][techsupport],
        "StreamingTV":     ENCODE["StreamingTV"][tv],
        "StreamingMovies": ENCODE["StreamingMovies"][movies],
        "Contract":        ENCODE["Contract"][contract],
        "PaperlessBilling":ENCODE["PaperlessBilling"][paperless],
        "PaymentMethod":   ENCODE["PaymentMethod"][payment],
        "MonthlyCharges":  float(monthly),
        "TotalCharges":    float(total),
    }

    # Pass a DataFrame to pipeline.predict() — exactly as Chapter 4 intended
    df_input = pd.DataFrame([row])
    result   = pipeline.predict(df_input, threshold=0.5)

    prob = float(result["churn_probability"].iloc[0])
    if prob >= 0.70:
        risk = "High risk  --  recommend a retention call immediately."
    elif prob >= 0.40:
        risk = "Medium risk  --  monitor and consider a proactive check-in."
    else:
        risk = "Low risk  --  customer appears stable."

    return f"{prob:.0%}", risk


# ── Gradio interface ────────────────────────────────────────────────────────────
demo = gr.Interface(
    fn = predict_churn,
    inputs = [
        gr.Dropdown(["Female", "Male"],                                         label="Gender"),
        gr.Dropdown(["No", "Yes"],                                              label="Senior Citizen"),
        gr.Dropdown(["No", "Yes"],                                              label="Partner"),
        gr.Dropdown(["No", "Yes"],                                              label="Dependents"),
        gr.Slider(0, 72, step=1, value=12,                                      label="Tenure (months)"),
        gr.Dropdown(["No", "Yes"],                                              label="Phone Service"),
        gr.Dropdown(["No", "No phone service", "Yes"],                         label="Multiple Lines"),
        gr.Dropdown(["DSL", "Fiber optic", "No"],                              label="Internet Service"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Online Security"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Online Backup"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Device Protection"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Tech Support"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Streaming TV"),
        gr.Dropdown(["No", "No internet service", "Yes"],                      label="Streaming Movies"),
        gr.Dropdown(["Month-to-month", "One year", "Two year"],                label="Contract"),
        gr.Dropdown(["No", "Yes"],                                              label="Paperless Billing"),
        gr.Dropdown(["Bank transfer (automatic)", "Credit card (automatic)",
                     "Electronic check", "Mailed check"],                      label="Payment Method"),
        gr.Slider(18, 120, step=0.5, value=65.0,                               label="Monthly Charges ($)"),
        gr.Number(value=780.0,                                                  label="Total Charges ($)"),
    ],
    outputs = [
        gr.Text(label="Churn Probability"),
        gr.Text(label="Risk Assessment"),
    ],
    title       = "Customer Churn Predictor",
    description = "Enter customer account details to predict churn likelihood.",
    examples    = [
        ["Male",   "No", "Yes", "No",  12, "Yes", "No",               "Fiber optic",
         "No", "No", "No", "No", "No", "No",
         "Month-to-month", "Yes", "Electronic check",          75.0,  900.0],
        ["Female", "No", "Yes", "Yes", 60, "Yes", "Yes",              "DSL",
         "Yes","Yes","Yes","Yes","Yes","Yes",
         "Two year",       "No",  "Bank transfer (automatic)", 45.0, 2700.0],
        ["Male",   "Yes","No",  "No",  2,  "Yes", "No phone service", "Fiber optic",
         "No", "No", "No", "No", "Yes","Yes",
         "Month-to-month", "Yes", "Electronic check",          95.0,  190.0],
    ],
    allow_flagging = "never",
)

print("\nLaunching local preview...")
demo.launch(share=False)


## Step 2 — Upload to Hugging Face Spaces

Once the local preview looks correct, uploading takes about 5 minutes.

---

### 1. Create a free Hugging Face account

Go to [huggingface.co](https://huggingface.co) and sign up.
No credit card. No personal information required beyond an email address.

---

### 2. Create a new Space

1. Click your profile picture → **New Space**
2. Fill in the form:

| Field | Value |
|-------|-------|
| Owner | your username |
| Space name | `churn-predictor` |
| License | MIT (or any) |
| **SDK** | **Gradio** ← important |
| Hardware | CPU Basic (free) |
| Visibility | Public or Private |

3. Click **Create Space**

---

### 3. Upload your three files

You will see an empty repository. Click **Add file → Upload files** and upload:

```
app.py              ← generated and downloaded by the cell above
churn_model_v1.pth  ← uploaded in the step above
requirements.txt    ← create this in the next cell
```

> **The file is already named `app.py`** — the cell above wrote it with that name
> and downloaded it to your computer. Upload it as-is.

---

### 4. Create `requirements.txt`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Create requirements.txt for the churn Space
#
# Hugging Face reads this file and installs these packages before running app.py.
# Keep versions pinned for reproducibility.
# ─────────────────────────────────────────────────────────────────────────────

req_lines = [
    "gradio==4.36.1",
    "torch==2.2.0",
    "numpy==1.26.4",
    "scikit-learn==1.4.2",
    "dill==0.3.8",
    "pandas==2.2.2",
]

with open("requirements.txt", "w") as f:
    f.write("\n".join(req_lines) + "\n")

print("requirements.txt written:")
for line in req_lines:
    print(f"  {line}")

from google.colab import files
files.download("requirements.txt")
print("requirements.txt downloaded -- upload this file to your Hugging Face Space.")

### 5. Watch the build and get your URL

After uploading all three files, Hugging Face automatically:
1. Detects the Gradio SDK
2. Installs the packages from `requirements.txt` (takes 2–4 minutes)
3. Runs `app.py`
4. Shows your live app at:

```
https://huggingface.co/spaces/your-username/churn-predictor
```

When the build finishes you will see a green **Running** badge.
Share that URL with anyone — they can use your churn model immediately.

> **If the build fails**, the Logs tab shows the error. The most common cause is
> a version mismatch in `requirements.txt`. Check that the torch version matches
> what was used to save the `.pth` file.

---
# B.4 Deploying the Defect Detector (CNN)

The CNN from Chapter 5 takes an **image** as input instead of a set of numbers.
Gradio handles image uploads natively — `gr.Image()` gives the user an upload
button, and Gradio passes the uploaded image directly to your function as a
PIL Image object.

The preprocessing inside `predict_defect()` mirrors exactly what was done
during training in Chapter 5: resize to 224×224, convert to tensor,
normalise with ImageNet mean and standard deviation.

## Before you start — upload your model file

The defect detector was saved as `defect_model_v1.pth` at the end of Chapter 5.
Upload it here before running any cells in this section.

> **To download from your Chapter 5 session:**
> ```python
> from google.colab import files
> files.download("defect_model_v1.pth")
> ```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Upload defect_model_v1.pth to this Colab session
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import files
import os

print("Click 'Choose Files' and select defect_model_v1.pth")
print()

uploaded = files.upload()

if "defect_model_v1.pth" in uploaded:
    size_mb = os.path.getsize("defect_model_v1.pth") / 1024 / 1024
    print(f"defect_model_v1.pth uploaded successfully  ({size_mb:.1f} MB)")
    print("You can now run the cells below.")
else:
    print("WARNING: the uploaded file is not named defect_model_v1.pth")
    print(f"  Files received: {list(uploaded.keys())}")
    print("  Rename the file to defect_model_v1.pth and re-run this cell.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write app.py for the defect detector CNN
# ─────────────────────────────────────────────────────────────────────────────

app_content = 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport numpy as np\nimport dill\nimport inspect\nimport gradio as gr\nfrom torchvision import transforms\n\nclass ModelPipeline:\n    def __init__(self):\n        self.model       = None\n        self.model_config = {}\n        self.class_names  = []\n\n    @classmethod\n    def load(cls, path):\n        ckpt   = torch.load(path, map_location=torch.device("cpu"), weights_only=False)\n        MC_raw = dill.loads(ckpt["model_class_bytes"])\n        try:\n            src = dill.source.getsource(MC_raw)\n            ns  = {"torch": torch, "nn": nn, "F": F, "np": np}\n            exec(compile(src, "<string>", "exec"), ns)\n            MC  = ns[MC_raw.__name__]\n        except Exception:\n            MC = MC_raw\n\n        valid = set(inspect.signature(MC.__init__).parameters) - {"self"}\n        cfg   = {k: v for k, v in ckpt["model_config"].items() if k in valid}\n\n        obj              = cls()\n        obj.model        = MC(**cfg)\n        obj.model.load_state_dict(ckpt["state_dict"])\n        obj.model.eval()\n        obj.model_config = ckpt.get("model_config", {})\n        obj.class_names  = ckpt.get("class_names", ["ok", "defect"])\n        return obj\n\nTRANSFORM = transforms.Compose([\n    transforms.Resize((224, 224)),\n    transforms.ToTensor(),\n    transforms.Normalize(mean=[0.485, 0.456, 0.406],\n                         std= [0.229, 0.224, 0.225]),\n])\n\npipeline    = ModelPipeline.load("defect_model_v1.pth")\nCLASS_NAMES = pipeline.class_names\n\ndef predict_defect(image):\n    if image is None:\n        return "No image uploaded.", {}\n    tensor = TRANSFORM(image).unsqueeze(0)\n    with torch.no_grad():\n        logits = pipeline.model(tensor)\n        probs  = torch.softmax(logits, dim=1).squeeze().numpy()\n    pred_idx   = int(np.argmax(probs))\n    pred_label = CLASS_NAMES[pred_idx]\n    confidence = float(probs[pred_idx])\n    if pred_label == "defect" and confidence >= 0.80:\n        verdict = f"DEFECT DETECTED  ({confidence:.0%} confidence)  -- flag for inspection."\n    elif pred_label == "defect":\n        verdict = f"Possible defect  ({confidence:.0%} confidence)  -- manual check recommended."\n    else:\n        verdict = f"No defect  ({confidence:.0%} confidence)  -- product OK."\n    scores = {name: float(p) for name, p in zip(CLASS_NAMES, probs)}\n    return verdict, scores\n\ndemo = gr.Interface(\n    fn      = predict_defect,\n    inputs  = gr.Image(type="pil", label="Upload product image"),\n    outputs = [\n        gr.Text(label="Verdict"),\n        gr.Label(label="Confidence scores"),\n    ],\n    title          = "Product Defect Detector",\n    description    = "Upload a product image to classify it as OK or defective.",\n    allow_flagging = "never",\n)\n\nif __name__ == "__main__":\n    demo.launch()\n'

with open("app.py", "w") as f:
    f.write(app_content)

print("app.py written and ready to upload to Hugging Face Spaces.")
print(f"  Lines: {len(app_content.splitlines())}")

# Download app.py directly to your computer
from google.colab import files
files.download("app.py")
print("app.py downloaded — upload this file to your HF Space.")

### Preview the defect detector locally

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Local preview of the defect detector
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import dill
import inspect
import gradio as gr
from torchvision import transforms

class ModelPipeline:
    def __init__(self):
        self.model       = None
        self.model_config = {}
        self.class_names  = []

    @classmethod
    def load(cls, path):
        ckpt   = torch.load(path, map_location=torch.device("cpu"), weights_only=False)
        MC_raw = dill.loads(ckpt["model_class_bytes"])
        try:
            src = dill.source.getsource(MC_raw)
            ns  = {"torch": torch, "nn": nn, "F": F, "np": np}
            exec(compile(src, "<string>", "exec"), ns)
            MC  = ns[MC_raw.__name__]
        except Exception:
            MC = MC_raw

        valid = set(inspect.signature(MC.__init__).parameters) - {"self"}
        cfg   = {k: v for k, v in ckpt["model_config"].items() if k in valid}

        obj              = cls()
        obj.model        = MC(**cfg)
        obj.model.load_state_dict(ckpt["state_dict"])
        obj.model.eval()
        obj.model_config = ckpt.get("model_config", {})
        obj.class_names  = ckpt.get("class_names", ["ok", "defect"])
        return obj

TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])

defect_pipeline = ModelPipeline.load("defect_model_v1.pth")
CLASS_NAMES = defect_pipeline.class_names
print(f"Pipeline loaded. Classes: {CLASS_NAMES}")

def predict_defect(image):
    if image is None:
        return "No image uploaded.", {}
    tensor = TRANSFORM(image).unsqueeze(0)
    with torch.no_grad():
        logits = defect_pipeline.model(tensor)
        probs  = torch.softmax(logits, dim=1).squeeze().numpy()
    pred_idx   = int(np.argmax(probs))
    pred_label = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])
    if pred_label == "defect" and confidence >= 0.80:
        verdict = f"DEFECT DETECTED  ({confidence:.0%} confidence)  -- flag for inspection."
    elif pred_label == "defect":
        verdict = f"Possible defect  ({confidence:.0%} confidence)  -- manual check recommended."
    else:
        verdict = f"No defect  ({confidence:.0%} confidence)  -- product OK."
    scores = {name: float(p) for name, p in zip(CLASS_NAMES, probs)}
    return verdict, scores

defect_demo = gr.Interface(
    fn      = predict_defect,
    inputs  = gr.Image(type="pil", label="Upload product image"),
    outputs = [
        gr.Text(label="Verdict"),
        gr.Label(label="Confidence scores"),
    ],
    title          = "Product Defect Detector",
    description    = "Upload a product image to classify it as OK or defective.",
    allow_flagging = "never",
)

print("Launching local preview...")
defect_demo.launch(share=False)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Create requirements.txt for the defect detector Space
# torchvision is needed for the image transforms
# ─────────────────────────────────────────────────────────────────────────────

req_lines = [
    "gradio==4.36.1",
    "torch==2.2.0",
    "torchvision==0.17.0",
    "numpy==1.26.4",
    "dill==0.3.8",
    "pillow==10.3.0",
]

with open("requirements.txt", "w") as f:
    f.write("\n".join(req_lines) + "\n")

print("requirements.txt written:")
for line in req_lines:
    print(f"  {line}")

from google.colab import files
files.download("requirements.txt")
print("requirements.txt downloaded -- upload this file to your Hugging Face Space.")
print()
print("Upload these three files to your 'defect-detector' Space:")
print("  app.py              (downloaded by the cell above)")
print("  defect_model_v1.pth (uploaded at the start of this section)")
print("  requirements.txt    (just downloaded)")

### 📝 Exercise B.4 — Deploy both apps

1. Deploy the churn predictor to a Space named `churn-predictor` (Section B.3 steps).
2. Deploy the defect detector to a Space named `defect-detector` (same steps, different files).
3. Open both live URLs and test them:
   - For the churn predictor: try the three example rows. Do the risk levels match your intuition?
   - For the defect detector: upload one of the casting defect images from the Chapter 5 dataset.
     Does the model classify it correctly?
4. Share one URL with someone who has not seen the notebook. Can they use the app without any explanation from you?

---
# B.5 Updating a Deployed Model

When you retrain a model and save a new `.pth` file, updating the live app
is a two-step process that takes about 2 minutes.

## The update workflow

```
Notebook                          Hugging Face Space
────────                          ──────────────────
retrain pipeline
     |
save as v2.pth
     |
go to Space in browser  ──────>  Files tab
                                      |
upload v2.pth                    replaces v1.pth
update app.py (one line)         rebuilds automatically
                                      |
test live URL          <──────── app is live with v2
```

---

## What changes in `app.py`

Exactly one line:

```python
# Before (app.py line 9)
pipeline = ModelPipeline.load("churn_model_v1.pth")

# After
pipeline = ModelPipeline.load("churn_model_v2.pth")
```

Upload the new `.pth` file and the updated `app.py` through the Files tab.
Hugging Face detects the change and rebuilds automatically.

---

## Version naming — same discipline as Chapters 3–6

| File | When |
|------|------|
| `churn_model_v1.pth` | Initial training (Chapter 4) |
| `churn_model_v2.pth` | Retrained on new monthly data |
| `churn_model_v3.pth` | Architecture change or major dataset update |

Never delete old `.pth` files from your Space. If `v2` performs worse than `v1`,
you roll back by uploading the old `app.py` that points to `v1.pth`.
One file upload. Done.

> **The same versioning discipline practised since Chapter 3 applies here.**
> Every `.pth` file is a checkpoint. Every version is a rollback option.

---
## Bonus Chapter Summary

| Concept | Key takeaway |
|---------|-------------|
| **The gap** | A notebook requires a human; a deployed app serves anyone automatically, 24/7 |
| **Gradio** | Turns a Python function into a web form; `gr.Interface()` handles all the UI |
| **Input components** | `gr.Slider()`, `gr.Number()`, `gr.Checkbox()` for tabular data; `gr.Image()` for CNN inputs |
| **Output components** | `gr.Text()` for strings; `gr.Label()` for class probabilities with a bar chart |
| **Image preprocessing** | The `transforms` pipeline inside `predict_defect()` must match what was used during training |
| **Hugging Face Spaces** | Permanent free hosting; upload `app.py`, `.pth`, `requirements.txt` → live URL |
| **requirements.txt** | Lists all packages the Space needs; pin versions for reproducibility |
| **Updating** | Upload new `.pth` → change one line in `app.py` → Space rebuilds automatically |
| **Rollback** | Reupload the old `app.py` pointing to the previous version — done in one file upload |

---

## The Complete Journey

```
Chapter 1  -->  A single neuron learns to fit a line
Chapter 2  -->  NumPy, pandas, PyTorch tensors
Chapter 3  -->  Training, saving, loading: the ModelPipeline
Chapter 4  -->  FFN: customer churn classifier
Chapter 5  -->  CNN: product defect detector
Chapter 6  -->  LSTM: CO2 demand forecaster
Chapter 7  -->  LLMs: earnings analyst bot
Bonus      -->  Sharing models as live web apps on Hugging Face Spaces
```

You started with a single artificial neuron. You finish with two live web apps
that anyone in the world can use from their browser.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*
*Dr. M. Ramasubramaniam & Mr. Daniel Peter*